# Load FACED EEG dataset and inspect shapes/labels

This notebook loads the FACED dataset using the `EEGDataset` implementation in `data/dataset_hdf5.py`, inspects sample shapes and labels, and runs basic sanity checks and a small visualization.


In [ ]:
from pathlib import Path
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import h5py

# Dataset directory (normalized to Linux-style slashes)
dataset_dir = Path("/mnt/dataset2/Processed_datasets/EEG_Bench/FACED_emo_label_smooth_window_2s").expanduser().resolve()
print("Dataset dir:", dataset_dir)
print("Exists:", dataset_dir.exists())
print("Example .h5 files:", sorted([p.name for p in dataset_dir.glob('*.h5')])[:10])


In [ ]:
# Add project data folder to path and import EEGDataset
proj_data_dir = Path('/mnt/dataset4/qingzhu/CBraMod/data').resolve()
if str(proj_data_dir) not in sys.path:
    sys.path.append(str(proj_data_dir))

from dataset_hdf5 import EEGDataset

# Instantiate dataset (this will compute per-subject stats and may print progress)
print('Instantiating EEGDataset...')
dataset = EEGDataset(dataset_dir)
print('Done instantiating.')


In [ ]:
# Inspect dataset metadata and preview samples
import pprint

print('Number of samples:', len(dataset))
print('Channel names (ch_names):', dataset.ch_names)
print('Dataset name:', dataset.dataset_name)
print('\nPreview of first 3 samples (file, path, meta):')
pp = pprint.PrettyPrinter(depth=2)
pp.pprint(dataset.samples[:3])


In [ ]:
# Load a single sample and display shapes, dtypes, and values
if len(dataset) == 0:
    print('Dataset is empty; nothing to load.')
else:
    data, class_label, rating_label, sub, trial = dataset[0]
    print('data.shape:', data.shape)
    print('data.dtype:', data.dtype)
    print('data.min(), data.max():', float(data.min()), float(data.max()))
    try:
        print('class_label.shape:', tuple(class_label.shape))
    except Exception:
        print('class_label:', class_label)
    try:
        print('rating_label.shape:', tuple(rating_label.shape))
    except Exception:
        print('rating_label:', rating_label)
    print('sub:', sub, 'trial:', trial)
    
    # Explanation: data layout -> [n_channels, n_segments, 200]
    print('\nNote: data layout is [n_channels, n_segments, 200] where 200 is the segment length.')


In [ ]:
# Batch-check labels and label distributions (up to 100 samples)
from collections import Counter

max_check = min(100, len(dataset))
class_tuples = []
rating_tuples = []
nan_count = 0

for i in range(max_check):
    data_i, class_i, rating_i, _, _ = dataset[i]
    # Convert to flat tuples for counting
    try:
        class_t = tuple(torch.flatten(class_i).cpu().numpy().tolist())
    except Exception:
        class_t = (str(class_i),)
    try:
        rating_t = tuple(torch.flatten(rating_i).cpu().numpy().tolist())
    except Exception:
        rating_t = (str(rating_i),)
    class_tuples.append(class_t)
    rating_tuples.append(rating_t)
    # Check NaNs
    if np.isnan(np.asarray(class_t)).any() or np.isnan(np.asarray(rating_t)).any():
        nan_count += 1

class_counts = Counter(class_tuples)
rating_counts = Counter(rating_tuples)

print('Checked samples:', max_check)
print('Samples with NaNs in labels:', nan_count)
print('Unique class-label vectors (up to 10 shown):', len(class_counts))
for i, (k, v) in enumerate(class_counts.most_common(10)):
    print(i+1, 'count=', v, 'sample_shape=', (len(k),))
    if i >= 4:
        break


In [ ]:
# Visualize first sample: first 4 channels of first segment
if len(dataset) > 0:
    data, _, _, _, _ = dataset[0]
    # data shape: [n_channels, n_segments, 200]
    n_channels = data.shape[0]
    seg0 = data[:, 0, :].cpu().numpy()  # shape [n_channels, 200]
    t = np.arange(seg0.shape[1])
    plt.figure(figsize=(10, 6))
    offset = 0
    for ch in range(min(4, n_channels)):
        plt.plot(t, seg0[ch] + offset, label=(dataset.ch_names[ch] if dataset.ch_names else f'ch{ch}'))
        offset += np.max(np.abs(seg0[ch])) * 2
    plt.xlabel('Sample index')
    plt.ylabel('Amplitude (offset for clarity)')
    plt.title('First sample — first 4 channels (segment 0)')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No samples to visualize')


In [ ]:
# Sanity checks: per-subject ranges and NaN assertions
print('Per-subject ranges (average min/max):')
try:
    ranges = dataset.calculate_sub_ranges()
except Exception as e:
    print('Error while calculating sub ranges:', e)

# Quick NaN and global min/max check on a subset
check_n = min(200, len(dataset))
global_min = float('inf')
global_max = float('-inf')
nan_found = False
for i in range(check_n):
    data_i, _, _, _, _ = dataset[i]
    arr = data_i.cpu().numpy()
    if np.isnan(arr).any():
        nan_found = True
        break
    global_min = min(global_min, float(arr.min()))
    global_max = max(global_max, float(arr.max()))

print('Checked samples for NaNs:', check_n)
print('NaN found in data subset:', nan_found)
print('Global min/max in subset:', global_min, global_max)


# How to run

# Run this notebook with Jupyter Lab or Notebook:
#   jupyter lab
# or
#   jupyter notebook

# To run all cells from the command line:
#   jupyter nbconvert --to notebook --execute notebooks/load_faced_dataset.ipynb --output notebooks/load_faced_dataset.executed.ipynb

print('Notebook ready. Run the cells to inspect the dataset.')
